<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-5-agents/Module_5_Session_3_LangGraph_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Session 3: Agents with LangGraph

## What are we building?
The same Swiggy ReAct agent from Session 2 — but rebuilt using
LangGraph. Instead of a manual loop, we define nodes (steps) and
edges (connections) and LangGraph manages the loop for us.

## Why does this matter?
Every production agent at FAANG scale uses a framework to manage
the agent loop. LangGraph is the industry standard. Understanding
it means you can build, debug, and extend real production agents.

## What we will learn:
- What a LangGraph node is (a Python function that does one job)
- What edges and conditional edges are (routing logic)
- How LangGraph manages state (replaces our manual history list)
- How the same ReAct loop looks inside a proper framework

## Model: openai/gpt-oss-20b on Groq
## New library: langgraph
## Platform: Google Colab

## Step 1: Install and Setup
We install langgraph — our one new library this session.
langchain-groq is needed so LangGraph can talk to our Groq model.

In [ ]:
# langgraph — new library this session
# langchain-groq — connects LangGraph to Groq API
!pip install langgraph langchain-groq langchain -q

# LangGraph imports
from langgraph.graph import StateGraph, END        # StateGraph is our graph builder, END is the exit node
from langchain_groq import ChatGroq                # LangChain wrapper for Groq
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage  # message types

# Standard imports
from typing import TypedDict, Annotated            # for defining our state structure
import operator                                     # for merging lists in state
import os
import json
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LangChain style Groq client — different from raw Groq client we used before
llm = ChatGroq(model="openai/gpt-oss-20b")

print("Setup complete")
print("LLM:", llm.model_name)

## Step 2: Define State
State is LangGraph's memory. It holds the full conversation history
as the graph moves from node to node. Every node reads from state
and writes back to state. This replaces our manual messages list
from Session 2.

In [ ]:
# State is a TypedDict — a dictionary with fixed keys and types
# Every node in our graph receives this state and returns an updated version

class AgentState(TypedDict):
    # Annotated[list, operator.add] means:
    # — messages is a list
    # — when two states are merged, lists are added together (not overwritten)
    # This is how conversation history grows automatically
    messages: Annotated[list, operator.add]

# Quick check — our state is just a container for messages
print("AgentState defined with keys:", list(AgentState.__annotations__.keys()))

## Step 3: Define Tools
Same three tools as Session 2.
One difference — we wrap them with @tool decorator so LangGraph
can understand and call them automatically.

In [ ]:
from langchain_core.tools import tool    # tool decorator — tells LangGraph this is a callable tool

# @tool decorator does two things:
# 1. Registers the function as a LangGraph tool
# 2. Uses the docstring as the tool description (replaces our long JSON schema from before)

@tool
def check_order_status(order_id: str) -> str:
    """Check the current status of a Swiggy order using the order ID.
    Use this first before deciding on refunds or any other action."""
    orders = {
        "ORD001": {"status": "Delayed", "eta": "45 minutes", "restaurant": "Burger King"},
        "ORD002": {"status": "Delivered", "eta": None, "restaurant": "Pizza Hut"},
        "ORD003": {"status": "Preparing", "eta": "25 minutes", "restaurant": "McDonald's"},
    }
    order = orders.get(order_id)
    if order:
        return f"Order {order_id} from {order['restaurant']}: {order['status']}. ETA: {order['eta']}"
    else:
        return f"Order {order_id} not found in system."

@tool
def calculate_refund(order_id: str, reason: str) -> str:
    """Calculate refund for a Swiggy order.
    Use this only after checking order status.
    Reason must be either wrong_item or late_delivery."""
    order_values = {"ORD001": 450, "ORD002": 320, "ORD003": 180}
    amount = order_values.get(order_id, 0)
    if reason == "wrong_item":
        refund = amount * 1.0
    elif reason == "late_delivery":
        refund = amount * 0.3
    else:
        refund = 0
    return f"Refund for {order_id}: ₹{refund} approved for reason: {reason}"

@tool
def escalate_to_human(order_id: str, reason: str) -> str:
    """Escalate the issue to a human support agent.
    Use this when you cannot resolve the issue with available tools."""
    return f"Order {order_id} escalated to human support. Reason: {reason}. Ticket created."

# Bind tools to LLM — LLM now knows these tools exist
tools = [check_order_status, calculate_refund, escalate_to_human]
llm_with_tools = llm.bind_tools(tools)

# Tool executor — maps tool name to function
tool_executor = {t.name: t for t in tools}

print("Tools registered:")
for t in tools:
    print(f"  - {t.name}")

## Step 4: Define Nodes
Nodes are the steps in our graph. Each node is a Python function
that takes state, does one job, and returns updated state.
We have two nodes:
1. agent_node — calls the LLM, decides what to do next
2. tool_node — runs whichever tool the LLM requested
This is exactly our Session 2 loop — split into two named functions.

In [ ]:
from langchain_core.messages import AIMessage   # message type for LLM responses

# --- NODE 1: Agent Node ---
# Job: send messages to LLM, get back a decision
# Either LLM calls a tool OR gives a final answer
def agent_node(state: AgentState):
    print("  [Agent Node] Thinking...")

    # Send full conversation history to LLM
    response = llm_with_tools.invoke(state["messages"])

    print(f"  [Agent Node] Tool calls: {len(response.tool_calls) if response.tool_calls else 0}")

    # Return new message — LangGraph appends it to state automatically
    return {"messages": [response]}

# --- NODE 2: Tool Node ---
# Job: run whichever tool the LLM requested, return result
def tool_node(state: AgentState):
    # Get the last message — it contains the tool call request from LLM
    last_message = state["messages"][-1]

    results = []

    # LLM may request multiple tools at once — handle all of them
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        print(f"  [Tool Node] Running: {tool_name}({tool_args})")

        # Run the tool
        tool_result = tool_executor[tool_name].invoke(tool_args)

        print(f"  [Tool Node] Result: {tool_result}")

        # Wrap result in ToolMessage — LangGraph requires this format
        results.append(ToolMessage(
            content=tool_result,
            tool_call_id=tool_call["id"]
        ))

    return {"messages": results}

print("Nodes defined:")
print("  - agent_node")
print("  - tool_node")

In [ ]:
# This function decides which node to go to after agent_node
# It is called a router or conditional edge function

def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    # If LLM made tool calls — go to tool_node
    if last_message.tool_calls:
        print("  [Router] Tool call detected → going to tool_node")
        return "tool_node"

    # If no tool calls — LLM gave final answer, go to END
    print("  [Router] No tool call → going to END")
    return END

# --- BUILD THE GRAPH ---
# Step 1: Create graph with our state definition
graph_builder = StateGraph(AgentState)

# Step 2: Add nodes
graph_builder.add_node("agent_node", agent_node)
graph_builder.add_node("tool_node", tool_node)

# Step 3: Set entry point — always start here
graph_builder.set_entry_point("agent_node")

# Step 4: Add conditional edge from agent_node
# After agent_node runs — call should_continue() to decide where to go
graph_builder.add_conditional_edges(
    "agent_node",       # from this node
    should_continue,    # call this function to decide
    {                   # mapping of return values to nodes
        "tool_node": "tool_node",
        END: END
    }
)

# Step 5: After tool_node always go back to agent_node
# This creates the loop — agent → tool → agent → tool → ... → END
graph_builder.add_edge("tool_node", "agent_node")

# Step 6: Compile the graph — makes it ready to run
graph = graph_builder.compile()

print("Graph compiled successfully")
print("\nGraph structure:")
print("  START → agent_node")
print("  agent_node → [tool call?] → tool_node → agent_node")
print("  agent_node → [no tool call?] → END")

## Step 6: Run the Graph
Now we run the compiled graph on real customer queries.
We just pass the initial message — LangGraph handles the rest.
No manual loop, no history management, no step counting.

In [ ]:
def run_langgraph_agent(customer_query):
    """
    Run our LangGraph Swiggy agent on a customer query.
    LangGraph manages the loop, history, and routing automatically.
    """
    print(f"Customer: {customer_query}")
    print("=" * 60)

    # Initial state — just the system message and customer query
    initial_state = {
        "messages": [
            SystemMessage(content="""You are a helpful Swiggy customer support agent.
            When a customer has a complaint, first check the order status, then take action.
            Only give a final answer when you have all the information you need."""),
            HumanMessage(content=customer_query)
        ]
    }

    # Run the graph — LangGraph handles everything from here
    final_state = graph.invoke(initial_state)

    # Final answer is the last message in state
    final_answer = final_state["messages"][-1].content

    print("=" * 60)
    print("Final answer to customer:")
    print(final_answer)
    print()
    return final_answer

# Test 1: Multi step — needs order check then refund
print("🔵 TEST 1: Late delivery refund")
run_langgraph_agent("My order ORD001 is very late, I want a refund please")

## Step 7: Test More Scenarios
Testing simple query and wrong item refund.
Same tests as Session 2 — so we can directly compare behaviour.

In [ ]:
# Test 2: Simple one step query
print("🟢 TEST 2: Simple status check")
run_langgraph_agent("What is the status of my order ORD003?")

# Test 3: Wrong item refund
print("🔴 TEST 3: Wrong item delivered")
run_langgraph_agent("I received the wrong item in order ORD002, I want a refund")

## Summary

### What we built:
The same Swiggy ReAct agent from Session 2 — rebuilt using LangGraph
with nodes, edges, and conditional routing.

### LangGraph concepts used:
- StateGraph — the graph builder
- AgentState — shared memory that flows between nodes
- agent_node — calls LLM, decides what to do next
- tool_node — runs the tool LLM requested
- should_continue — conditional edge, routes to tool_node or END
- add_edge — fixed connection between nodes
- add_conditional_edges — dynamic routing based on LLM decision
- compile() — locks the graph and makes it ready to run

### Session 2 vs Session 3:
| Session 2 (raw Python) | Session 3 (LangGraph) |
|---|---|
| Manual for loop | Graph loop managed by LangGraph |
| Manual messages.append() | operator.add merges automatically |
| Manual if/else routing | Conditional edges |
| Manual tool dispatch dict | @tool decorator + bind_tools |
| 60 lines of loop code | 15 lines with LangGraph |
| Hard to extend | Add a new node and edge |

### Key interview insight:
LangGraph does not make your agent smarter.
It makes your agent code cleaner, more maintainable, and easier to debug.
The intelligence still comes from the LLM and your tool descriptions.

### AWS Equivalent:
| This session | AWS |
|---|---|
| StateGraph | AWS Step Functions state machine |
| agent_node | Lambda function — reasoning step |
| tool_node | Lambda function — action step |
| conditional edge | Step Functions choice state |
| END | Step Functions success state |
| Full graph | Amazon Bedrock Agents (all of this built in) |

### Next session:
Module 5 Session 4 — Agent Memory
We give the agent memory so it remembers the customer and
conversation across multiple turns — like a real support agent.